In [ ]:
'''
python version 3.10.12
'''

In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install requirements.txt
'''
Please restart this file after executing this cell !!!
'''

'\nPlease restart this file after executing this cell !!!\n'

In [3]:
import argparse
import sys
import os
import data_utils
import numpy as np
from torch import Tensor
from torch.utils.data import DataLoader
from torchvision import transforms
import yaml
import torch
from torch import nn
from model import RawGAT_ST  # In main model script we used our best RawGAT-ST-mul model. To use other models you need to call revelant model scripts from RawGAT_models folder
# from tensorboardX import SummaryWriter
from core_scripts.startup_config import set_random_seed
import time
from tqdm import tqdm

In [4]:



def pad(x, max_len=64600):
    x_len = x.shape[0]
    if x_len >= max_len:
        return x[:max_len]
    # need to pad
    num_repeats = int(max_len / x_len)+1
    padded_x = np.tile(x, (1, num_repeats))[:, :max_len][0]
    return padded_x


def evaluate_accuracy(data_loader, model, device):
    val_loss = 0.0
    num_total = 0.0
    model.eval()

    
    weight = torch.FloatTensor([0.1, 0.9]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)

    for batch_x, batch_y, batch_meta in data_loader:
        
        batch_size = batch_x.size(0)
        num_total += batch_size
        
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        
        batch_out = model(batch_x,Freq_aug=False)
        
        batch_loss = criterion(batch_out, batch_y)
        val_loss += (batch_loss.item() * batch_size)
        
    val_loss /= num_total
   
    return val_loss


def produce_evaluation_file(eval_num,dataset, model, device, save_path):
    data_loader = DataLoader(dataset, batch_size=24, shuffle=False)
    num_correct = 0.0
    num_total = 0.0
    model.eval()
    
    fname_list = []
    key_list = []
    variant_list = []
    random_list=[]
    src_list=[]
    score_list = []
 
    print('start eva')
    for batch_x, batch_y, batch_meta in tqdm(data_loader):
        
        batch_size = batch_x.size(0)
        num_total += batch_size
        
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        batch_out = model(batch_x,Freq_aug=False)
        
        batch_score = (batch_out[:, 1]  
                       ).data.cpu().numpy().ravel()     
        

        # add outputs
        fname_list.extend(list(batch_meta[1]))
        variant_list.extend(list(batch_meta[0]))
        random_list.extend(list(batch_meta[2]))
        src_list.extend(list(batch_meta[3]))
        key_list.extend(
          ['bonafide' if key == 1 else 'spoof' for key in list(batch_meta[4])])
        # sys_id_list.extend([dataset.sysid_dict_inv[s.item()]
        #                     for s in list(batch_meta[3])])
        score_list.extend(batch_score.tolist())

    with open(save_path, 'w') as fh:
        for v,f, r,s, k, cm in zip(variant_list,fname_list, random_list,src_list, key_list, score_list):
            # if dataset.is_eval:
            fh.write('{} {} {} {} {} {}\n'.format(v,f, r,s, k, cm))
            # else:
                # fh.write('{} {}\n'.format(f, cm))
    print('Result saved to {}'.format(save_path))

def train_epoch(data_loader, model, lr,optimizer, device):
    running_loss = 0
    num_total = 0.0
    model.train()

    # set objective (Loss) functions --> WCE
    weight = torch.FloatTensor([0.1, 0.9]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    
    for batch_x, batch_y, batch_meta in data_loader:
        
        batch_size = batch_x.size(0)

        num_total += batch_size
        
        batch_x = batch_x.to(device)
       
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        
        
        batch_out = model(batch_x,Freq_aug=True)
        
        batch_loss = criterion(batch_out, batch_y)
        
        running_loss += (batch_loss.item() * batch_size)
       
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()
       
    running_loss /= num_total
    
    return running_loss

In [ ]:
dir_yaml = os.path.splitext('model_config_RawGAT_ST')[0] + '.yaml'

with open(dir_yaml, 'r') as f_yaml:
        # parser1 = yaml.load(f_yaml)
        parser1 = yaml.safe_load(f_yaml)

if not os.path.exists('models'):
    os.mkdir('models')
# args = parser.parse_args()

#make experiment reproducible
# set_random_seed(args.seed, args)
set_random_seed(1234)

# track = args.track
track='logical'
assert track in ['logical', 'physical'], 'Invalid track given'
is_logical = (track == 'logical')

transforms = transforms.Compose([
    lambda x: pad(x),
    lambda x: Tensor(x)
    
])


#GPU device
device = 'cuda:1' if torch.cuda.is_available() else 'cpu'                  
print('Device: {}'.format(device))
'''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
'''
database_path='change this to your VoiceWukong dataset path' 
protocols_path1='change this to eval_list.txt'
protocols_path2='change this to zh_eval_list.txt'
features='Raw_GAT'
lss='WCE'
# validation Dataloader
is_eval=True
eval_part=0

dev_set1 = data_utils.ASVDataset(database_path=database_path,protocols_path=protocols_path1,is_train=False, is_logical=is_logical,
                                transform=transforms,
                                feature_name=features, is_eval=True, eval_part=eval_part)
dev_set2 = data_utils.ASVDataset(database_path=database_path,protocols_path=protocols_path2,is_train=False, is_logical=is_logical,
                                transform=transforms,
                                feature_name=features, is_eval=True, eval_part=eval_part)
model = RawGAT_ST(parser1['model'], device)
nb_params = sum([param.view(-1).size()[0] for param in model.parameters()])
model =(model).to(device)

# Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001,weight_decay=0.0001)

model_path='change this to RawGAT-ST model path '# download here [https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/RawGAT.pth?download=true]

# if args.model_path:
model.load_state_dict(torch.load(model_path,map_location=device))
print('Model loaded : {}'.format(model_path))



produce_evaluation_file(1,dev_set1, model, device,'change this to the path to save en_eval_score.txt')
produce_evaluation_file(2,dev_set2, model, device,'change this to the path to save zh_eval_score.txt')
 